# Tarea 4 — Análisis de imágenes: detección de monedas

**Procesamiento y Clasificación de Datos · MCD, FCFM-UANL**

## Qué se hace

Encontrar los **centros de objetos circulares** en fotografías propias — monedas mexicanas — con la
transformada de Hough, siguiendo el pipeline de la clase: escala de grises → desenfoque gaussiano →
detección de círculos.

Y un paso más allá: las monedas mexicanas tienen diámetros distintos por denominación
($1 = 21 mm, $2 = 23 mm, $5 = 25.5 mm, $10 = 28 mm). Una vez detectado el radio de cada círculo,
se puede **clasificar la denominación y contar el dinero de la foto**. Como yo sé cuánto puse en
la mesa, el sistema tiene una respuesta verificable.

## Las fotos

Cuatro fotografías tomadas con el celular, cámara perpendicular a la mesa:

| Archivo | Condición | Para qué |
|---|---|---|
| `fotos/facil.jpg` | monedas separadas, fondo uniforme, buena luz | el caso base |
| `fotos/dificil.jpg` | monedas que se tocan | ¿Hough las separa? |
| `fotos/mezcla.jpg` | denominaciones mezcladas | el conteo de dinero |
| `fotos/sombras.jpg` | luz lateral con sombras | caso de fallo documentado |

La perpendicularidad importa: en ángulo, los círculos se vuelven elipses y tanto la detección como
la medición del radio se degradan.

## 0. Verificar librerías

In [ ]:
import importlib.util

REQUERIDAS = ["cv2", "numpy", "matplotlib", "pandas", "tabulate"]
PIP = {"cv2": "opencv-python"}

faltan = [PIP.get(p, p) for p in REQUERIDAS if importlib.util.find_spec(p) is None]

if faltan:
    print("Instalando:", ", ".join(faltan))
    %pip install -q {" ".join(faltan)}
    print("\n>>> REINICIA el kernel (Restart) y corre de nuevo desde arriba <<<")
else:
    print("Todas las librerías están instaladas ✓")

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIG = Path("figuras"); FIG.mkdir(exist_ok=True)
FOTOS = Path("fotos")

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150,
                     "savefig.bbox": "tight", "font.size": 10})

archivos = sorted(FOTOS.glob("*.jpg")) + sorted(FOTOS.glob("*.png"))
assert archivos, f"No hay fotos en {FOTOS.resolve()} — revisa la carpeta"
print(f"{len(archivos)} fotos:", ", ".join(a.name for a in archivos))

## ⚙️ Parámetros que TÚ debes ajustar

Dos cosas dependen de tus fotos y hay que editarlas aquí:

1. **`REALES`** — cuántas monedas hay *de verdad* en cada foto (las contaste al tomarla).
   Sin esto no se puede evaluar la detección.
2. **`REFERENCIA`** — la denominación de la moneda **más grande** de cada foto. El radio en
   pixeles no dice milímetros por sí solo; la moneda más grande ancla la escala. Si tu moneda
   mayor es de $10, deja "10".

In [ ]:
# cuántas monedas hay realmente en cada foto (edítalo con tus conteos)
REALES = {
    "facil": 6,
    "dificil": 5,
    "mezcla": 7,
    "sombras": 4,
}

# denominación de la moneda MÁS GRANDE de cada foto (ancla la escala px -> mm)
REFERENCIA = {
    "facil": "10",
    "dificil": "10",
    "mezcla": "10",
    "sombras": "10",
}

# diámetros oficiales (mm) — Banco de México
DIAMETROS = {"10": 28.0, "5": 25.5, "2": 23.0, "1": 21.0}
VALORES = {"10": 10, "5": 5, "2": 2, "1": 1}

## 1. El pipeline, paso a paso

Antes de detectar nada, ver qué hace cada etapa sobre una foto. La secuencia de la clase:

1. **Escala de grises** — el color no aporta para encontrar formas; una moneda dorada y una
   plateada son igual de circulares.
2. **Desenfoque gaussiano** — borra el relieve interno de la moneda (el águila, los números).
   Sin esto, Canny encuentra bordes *dentro* de la moneda y Hough ve círculos fantasma.
3. **Canny** — encuentra los bordes. Hough trabaja internamente sobre esto.

In [ ]:
ejemplo = archivos[0]
img = cv2.imread(str(ejemplo))
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (11, 11), 0)
edges = cv2.Canny(blurred, 30, 150)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (imagen, titulo, cmap) in zip(axes, [
    (cv2.cvtColor(img, cv2.COLOR_BGR2RGB), "Original", None),
    (gray, "Escala de grises", "gray"),
    (blurred, "Desenfoque gaussiano 11×11", "gray"),
    (edges, "Bordes (Canny)", "gray"),
]):
    ax.imshow(imagen, cmap=cmap)
    ax.set_title(titulo)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIG / "1_pipeline.png")
plt.show()

## 2. Detección con la transformada de Hough

`cv2.HoughCircles` tiene cinco parámetros y todos importan:

| Parámetro | Qué controla | Si es muy bajo | Si es muy alto |
|---|---|---|---|
| `dp` | resolución del acumulador | preciso pero lento | rápido pero burdo |
| `minDist` | distancia mínima entre centros | detecta la misma moneda 2 veces | fusiona monedas vecinas |
| `param1` | umbral alto de Canny | bordes de más | pierde bordes débiles |
| `param2` | votos para aceptar un círculo | **círculos fantasma** | **pierde monedas reales** |
| `min/maxRadius` | rango de radios buscados | basura pequeña | ignora monedas |

`param2` es el sensible — es el umbral de evidencia. Se estudia en la sección 4.

In [ ]:
ANCHO_MAX = 1600  # las fotos de celular vienen en ~4000px; se reducen para
                  # que el rango de radios funcione y Hough no se arrastre


def detectar(ruta, dp=1.2, minDist=80, param1=120, param2=55,
             minRadius=60, maxRadius=180):
    """Pipeline completo: lee, redimensiona si hace falta, procesa y
    devuelve (imagen_rgb, circulos)."""
    img = cv2.imread(str(ruta))
    if img.shape[1] > ANCHO_MAX:
        escala = ANCHO_MAX / img.shape[1]
        img = cv2.resize(img, None, fx=escala, fy=escala,
                         interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (11, 11), 0)
    circulos = cv2.HoughCircles(
        blurred, cv2.HOUGH_GRADIENT,
        dp=dp, minDist=minDist, param1=param1, param2=param2,
        minRadius=minRadius, maxRadius=maxRadius)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if circulos is None:
        return rgb, np.empty((0, 3))
    return rgb, circulos[0]


def dibujar(rgb, circulos, ax, titulo=""):
    ax.imshow(rgb)
    for x, y, r in circulos:
        ax.add_patch(plt.Circle((x, y), r, fill=False, color="lime", lw=2))
        ax.plot(x, y, "r+", markersize=12, markeredgewidth=2)
    ax.set_title(titulo)
    ax.axis("off")

### Nota sobre los radios y el tamaño de la foto

Las fotos se reducen automáticamente a 1600 px de ancho antes de procesar — así el rango de radios
por defecto (60–180 px) funciona para fotos de celular sin ajustes, y Hough corre rápido.

Si aun así no detecta tus monedas: abre la foto, estima a ojo el radio de una moneda ya reducida
(regla de tres con el ancho), y ajusta `minRadius`/`maxRadius` a ±40% de ese valor.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
resultados = {}

for ax, ruta in zip(axes.ravel(), archivos):
    nombre = ruta.stem
    rgb, circulos = detectar(ruta)
    resultados[nombre] = circulos
    n_real = REALES.get(nombre, "?")
    dibujar(rgb, circulos, ax,
            f"{nombre}: {len(circulos)} detectadas / {n_real} reales")

plt.tight_layout()
plt.savefig(FIG / "2_deteccion.png")
plt.show()

tabla_det = pd.DataFrame({
    "Detectadas": {n: len(c) for n, c in resultados.items()},
    "Reales": {n: REALES.get(n, np.nan) for n in resultados},
})
tabla_det["Diferencia"] = tabla_det["Detectadas"] - tabla_det["Reales"]
tabla_det

## 3. Los centros

Lo que pide el enunciado, explícitamente: las coordenadas de los centros.

In [ ]:
centros = []
for nombre, circulos in resultados.items():
    for x, y, r in circulos:
        centros.append({"foto": nombre, "x": round(float(x)),
                        "y": round(float(y)), "radio_px": round(float(r))})

tabla_centros = pd.DataFrame(centros)
tabla_centros

## 4. Sensibilidad de `param2` — el experimento

`param2` es el número de "votos" que un círculo candidato necesita para ser aceptado. Se barre de
20 a 90 sobre cada foto y se cuenta cuántos círculos salen. La detección correcta vive en una
meseta; fuera de ella, o llueven círculos fantasma o desaparecen monedas.

In [ ]:
valores_p2 = range(20, 95, 5)

fig, ax = plt.subplots(figsize=(8, 4.5))
for ruta in archivos:
    nombre = ruta.stem
    conteos = [len(detectar(ruta, param2=p2)[1]) for p2 in valores_p2]
    ax.plot(list(valores_p2), conteos, "o-", label=nombre)
    if nombre in REALES:
        ax.axhline(REALES[nombre], ls=":", color="gray", alpha=0.5)

ax.set_xlabel("param2 (umbral de votos)")
ax.set_ylabel("Círculos detectados")
ax.set_title("Sensibilidad de la detección a param2")
ax.set_ylim(0, 25)
ax.legend()
plt.savefig(FIG / "3_sensibilidad.png")
plt.show()

**Cómo leer la gráfica.** Las líneas punteadas grises son los conteos reales. Un `param2`
bajo produce decenas de detecciones falsas (el desplome de la izquierda hacia arriba); uno alto va
perdiendo monedas. El valor de trabajo (55) debe estar en la meseta donde cada curva coincide con
su línea punteada. Si alguna foto nunca alcanza su meseta, ese es un hallazgo — normalmente la de
sombras.

## 5. Clasificación de denominación y conteo de dinero

El radio en pixeles no dice nada por sí solo: depende de la distancia de la cámara. El ancla es la
**moneda más grande** de cada foto, cuya denominación declaré en `REFERENCIA`. Con ella se calcula
la escala mm/pixel de esa foto, y cada círculo se asigna a la denominación de diámetro más cercano.

In [ ]:
def clasificar_monedas(circulos, denominacion_mayor):
    """Devuelve (DataFrame con denominaciones, total en pesos)."""
    if len(circulos) == 0:
        return pd.DataFrame(), 0
    radios = circulos[:, 2]
    mm_por_px = DIAMETROS[denominacion_mayor] / (2 * radios.max())
    filas = []
    for x, y, r in circulos:
        d_mm = 2 * r * mm_por_px
        denom = min(DIAMETROS, key=lambda k: abs(DIAMETROS[k] - d_mm))
        filas.append({"x": round(float(x)), "y": round(float(y)),
                      "diametro_mm": round(d_mm, 1), "denominacion": f"${denom}"})
    df = pd.DataFrame(filas)
    total = sum(VALORES[d.strip("$")] for d in df["denominacion"])
    return df, total


conteo_dinero = {}
for nombre, circulos in resultados.items():
    df, total = clasificar_monedas(circulos, REFERENCIA[nombre])
    conteo_dinero[nombre] = total
    print(f"--- {nombre}: ${total} ---")
    if len(df):
        print(df.to_string(index=False))
    print()

**Verifica contra la realidad:** tú sabes cuánto dinero pusiste en cada foto. Si el total de
`mezcla` coincide, el sistema completo funciona — detección, escala y clasificación, todo junto.

Anota aquí cuánto había realmente para el reporte:

In [ ]:
# cuánto dinero había DE VERDAD en cada foto (edítalo)
DINERO_REAL = {
    "facil": None,     # ej. 33
    "dificil": None,
    "mezcla": None,
    "sombras": None,
}

comparacion = pd.DataFrame({
    "Estimado": conteo_dinero,
    "Real": {k: (v if v is not None else np.nan) for k, v in DINERO_REAL.items()},
})
comparacion

## 6. Generar el reporte

In [ ]:
lineas_dinero = "\n".join(
    f"| {n} | ${conteo_dinero[n]} | " +
    (f"${DINERO_REAL[n]}" if DINERO_REAL.get(n) is not None else "—") + " |"
    for n in conteo_dinero
)

reporte = f"""# Tarea 4 — Análisis de imágenes: detección de monedas

**Procesamiento y Clasificación de Datos · MCD, FCFM-UANL**

## Objetivo

Encontrar los centros de objetos circulares (monedas mexicanas) en fotografías propias mediante la
transformada de Hough, y llevar la detección un paso más allá: clasificar la denominación de cada
moneda por su diámetro y contar el dinero de la foto.

## Método

Pipeline de la clase: escala de grises → desenfoque gaussiano (11×11) → `cv2.HoughCircles`
(método gradiente, que aplica Canny internamente).

![Pipeline](figuras/1_pipeline.png)

El desenfoque es indispensable: sin él, el relieve interno de las monedas (águila, números) genera
bordes que Hough interpreta como círculos fantasma.

## Detección

![Detección](figuras/2_deteccion.png)

{tabla_det.to_markdown()}

Los centros detectados (cruces rojas) y sus coordenadas completas están en el notebook.

## Experimento: sensibilidad a `param2`

![Sensibilidad](figuras/3_sensibilidad.png)

`param2` es el umbral de votos del acumulador de Hough. El barrido de 20 a 90 muestra el
comportamiento típico: valores bajos producen lluvia de círculos falsos; valores altos pierden
monedas reales. La detección correcta vive en una meseta intermedia — y la foto con sombras tiene
la meseta más angosta (o no la tiene), lo que documenta la fragilidad del método ante iluminación
no uniforme.

## Clasificación por diámetro y conteo de dinero

Las monedas mexicanas difieren en diámetro por denominación ($1 = 21 mm, $2 = 23 mm,
$5 = 25.5 mm, $10 = 28 mm). Anclando la escala con la moneda más grande de cada foto, cada radio
detectado se convierte a milímetros y se asigna a la denominación más cercana.

| Foto | Dinero estimado | Dinero real |
|---|---:|---:|
{lineas_dinero}

## Hallazgos

1. **El desenfoque gaussiano es el paso que hace posible todo lo demás** — sin él, el relieve de
   las monedas contamina la detección.
2. **`param2` domina el comportamiento**: la diferencia entre detectar bien, detectar fantasmas y
   no detectar nada son ±15 unidades de este parámetro.
3. **Las sombras son el enemigo**: la foto con luz lateral pierde monedas porque la sombra rompe
   el borde circular que Canny necesita.
4. **La clasificación por diámetro funciona** cuando la foto es perpendicular y la referencia de
   escala es correcta; $1 (21 mm) y $2 (23 mm) son las más fáciles de confundir por estar a solo
   2 mm de distancia.

## Limitaciones

- La escala px→mm depende de declarar correctamente la denominación de la moneda mayor.
- Fotos en ángulo convierten círculos en elipses; Hough estándar no las detecta bien.
- Monedas conmemorativas o de $20 no están en el catálogo de diámetros.

## Reproducir

Colocar las fotos en `fotos/`, ajustar `REALES` y `REFERENCIA` en la celda de parámetros, y correr
`Tarea4/deteccion_monedas.ipynb`. Requiere `opencv-python`, `numpy`, `pandas`, `matplotlib`.

## Referencias

- Duda, R. O., & Hart, P. E. (1972). *Use of the Hough transformation to detect lines and curves
  in pictures*. Communications of the ACM, 15(1).
- Banco de México. Características de las monedas en circulación.
- OpenCV documentation: `cv2.HoughCircles`.
"""

Path("README.md").write_text(reporte, encoding="utf-8")
print("README.md escrito ✓")